In [ ]:
# =========================
#version_6
# N-Hits model
#loss -> Huber(weighted) + smape 
#Early stopping
# 예측 값이 spike를 못잡음
# =========================

import os
import random
import glob
import re

import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler

import torch
import torch.nn as nn
from tqdm import tqdm
import torch.nn.functional as F

import matplotlib.pyplot as plt
from korean_lunar_calendar import KoreanLunarCalendar

from copy import deepcopy

from collections import defaultdict



plt.rcParams['font.family'] = 'AppleGothic'  # macOS
plt.rcParams['axes.unicode_minus'] = False  # 하이픈으로 대체


########### Util ################
class EarlyStopping:
    def __init__(self, patience=10, min_delta=0.0, mode='min', restore_best_weights=True):
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.restore_best_weights = restore_best_weights
        self.best = None
        self.best_state = None
        self.wait = 0
        self.stop = False

    def step(self, current, model):
        if self.best is None:
            self.best = current
            self.best_state = deepcopy(model.state_dict())
            self.wait = 0
            return False

        improved = (current < self.best - self.min_delta) if self.mode == 'min' else (current > self.best + self.min_delta)

        if improved:
            self.best = current
            self.best_state = deepcopy(model.state_dict())
            self.wait = 0
        else:
            self.wait += 1
            if self.wait >= self.patience:
                self.stop = True
                if self.restore_best_weights and self.best_state is not None:
                    model.load_state_dict(self.best_state)
                return True
        return False
    
def add_ts_stats(
    df: pd.DataFrame,
    target_col: str = "clipped_SQ",   # 스케일 전(or 스케일 후) 타깃 중 택1
    date_col: str = "영업일자",
    lags = (1, 7, 14, 28),
    roll_windows = (7, 14, 28),
    ewm_spans = (7, 14),
    out_prefix: str = "",
    eps: float = 1e-3
) -> pd.DataFrame:
    """
    시계열 통계 피처 생성 (모두 과거만 사용)
    생성: lag_k, roll_mean_k, roll_std_k, ewm_mean_s, momentum_k, rel_level_k, vol_k
    - momentum_k  = (x_t - x_{t-k}) / (|x_{t-k}|+eps)
    - rel_level_k = x_t / (roll_mean_k + eps)
    - vol_k       = roll_std_k / (roll_mean_k + eps)
    """
    # 정렬 보장
    df = df.sort_values(date_col)
    x = df[target_col].astype("float32")

    # Lags
    for k in lags:
        df[f"{out_prefix}lag_{k}"] = x.shift(k).astype("float32")

    # Rolling mean/std (과거 window, 현재 포함 → 누수 방지 위해 shift(1) 후 rolling도 가능)
    for w in roll_windows:
        # 현재 시점 포함 롤링 → 일반적으로 OK. 더 엄격히 하려면 아래 두 줄을 교체:
        #    base = x.shift(1) ; df[f"roll_mean_{w}"] = base.rolling(w, min_periods=1).mean()
        df[f"{out_prefix}roll_mean_{w}"] = x.rolling(w, min_periods=1).mean().astype("float32")
        df[f"{out_prefix}roll_std_{w}"]  = x.rolling(w, min_periods=1).std().fillna(0).astype("float32")

    # EWMA
    for s in ewm_spans:
        df[f"{out_prefix}ewm_mean_{s}"] = x.ewm(span=s, adjust=False).mean().astype("float32")

    # Momentum & Relative level & Volatility (대표 window=7 사용; 필요시 반복문 확장)
    for k in lags:
        df[f"{out_prefix}momentum_{k}"] = ((x - x.shift(k)) / (np.abs(x.shift(k)) + eps)).astype("float32")
    for w in roll_windows:
        m = df[f"{out_prefix}roll_mean_{w}"]
        s = df[f"{out_prefix}roll_std_{w}"]
        df[f"{out_prefix}rel_level_{w}"] = (x / (m + eps)).astype("float32")      # 수준/평균
        df[f"{out_prefix}vol_{w}"]       = (s / (m + eps)).astype("float32")      # 변동성/평균(무단위)

    return df
#################################

#Fixed Random Seed  & Setting Hyperparameter
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)


set_seed(42)

LOOKBACK, PREDICT, BATCH_SIZE, EPOCHS = 28, 7, 32, 50
DEVICE = torch.device('mps' if torch.backends.mps.is_available() else
                      'cuda' if torch.cuda.is_available() else 'cpu')
MONTH_SCALE = 12

MIN_SEQUENCE_COUNT = 10

USE_SMAPE = True
PATIENCE = 10
MIN_DELTA = 0.0
MAX_LAG = 28

# 키: '영업장명_메뉴명', 값: 'YYYY-MM-DD' (ISO 문자열 또는 datetime)
DISCONTINUED = {
    '담하_꼬막_비빔밥': '2024-04-01',
    '담하_들깨_양지탕': '2024-04-01',
    # ...
}

def get_lunar_to_solar(years, lunar_month, lunar_day, span=1):
    calendar = KoreanLunarCalendar()
    dates = []
    for year in years:
        for offset in range(-span, span+1):
            try:
                calendar.setLunar(year, lunar_month, lunar_day + offset, False)
                dates.append(calendar.SolarIsoFormat())
            except:
                pass  # 예외 처리: 음력 마지막날 초과
    return dates
# 예시: 2023 ~ 2025
years = [2023, 2024, 2025]
lunar_solar_dates = []
lunar_solar_dates += get_lunar_to_solar(years, 1, 1, span=1)   # 설날 ±1
lunar_solar_dates += get_lunar_to_solar(years, 8, 15, span=1)  # 추석 ±1

solar_md_holidays = [
    (1, 1),   # 신정
    (3, 1),   # 삼일절
    (5, 5),   # 어린이날
    (6, 6),   # 현충일
    (8, 15),  # 광복절
    (10, 3),  # 개천절
    (10, 9),  # 한글날
    (12, 25), # 크리스마스
]

def generate_combined_holiday_list(df, solar_md_list, lunar_solar_list):
    df = df.copy()
    df['영업일자'] = pd.to_datetime(df['영업일자'])

    # 양력 기반 holiday 판별
    df['is_solar_holiday'] = df['영업일자'].apply(
        lambda x: (x.month, x.day) in solar_md_list
    )

    # 음력 변환된 holiday 포함
    lunar_set = set(pd.to_datetime(lunar_solar_list))
    df['is_lunar_holiday'] = df['영업일자'].isin(lunar_set)

    # 최종 통합
    df['is_holiday'] = (df['is_solar_holiday'] | df['is_lunar_holiday']).astype(int)
    df = df.drop(columns=['is_solar_holiday', 'is_lunar_holiday'])
    return df

def remove_leading_zeros_before_sales(df, min_zero_days=90):
    """
    매출 시작 전 연속 0이 일정 기간 이상이면, 그 전 구간 제거
    (단일 메뉴-업장 그룹 DataFrame을 가정)
    """
    sales_started = df['매출수량'] > 0
    if not sales_started.any():
        return df  # 매출이 전혀 없는 경우 그대로 반환

    first_sale_idx = sales_started.idxmax()

    # 매출 시작 전 구간이 충분히 긴 0으로 구성되어 있다면 제거
    df_before = df.loc[:first_sale_idx - 1]
    if len(df_before) >= min_zero_days and (df_before['매출수량'] == 0).all():
        return df.loc[first_sale_idx:]  # 매출 시작부터 반환
    else:
        return df  # 그대로 반환


def _extract_store_name(g: pd.DataFrame) -> str:
    """
    그룹 g에서 업장명 추출:
    - '영업장명' 컬럼이 있으면 그 값을 사용
    - 없으면 '영업장명_메뉴명'에서 첫 '_' 앞을 업장명으로 간주
    """
    if '영업장명' in g.columns:
        return str(g['영업장명'].iloc[0])
    # '영업장명_메뉴명'이 "업장명_메뉴명" 형태라고 가정
    full = str(g['영업장명_메뉴명'].iloc[0])
    return full.split('_', 1)[0]  # '_'가 여러 개여도 첫 구분만 사용


def filter_all_menus_by_leading_zeros(
    train_df: pd.DataFrame,
    min_zero_days: int = 90,
    apply_to_stores: list[str] | None = None,
    exclude_stores: list[str] | None = None,
    group_col: str = '영업장명_메뉴명',
) -> pd.DataFrame:
    """
    모든 메뉴-업장 그룹에 대해 remove_leading_zeros_before_sales를 적용하되,
    특정 업장에만(또는 특정 업장은 제외하고) 적용할 수 있도록 확장.

    Parameters
    ----------
    train_df : 전체 데이터프레임
    min_zero_days : 매출 시작 전 연속 0 최소 일수
    apply_to_stores : 적용 대상 업장명 리스트 (None이면 전 업장 대상)
    exclude_stores : 적용 제외 업장명 리스트 (None이면 제외 없음)
    group_col : 그룹화 기준 컬럼명 (기본: '영업장명_메뉴명')
    """
    parts = []
    apply_set   = set(apply_to_stores) if apply_to_stores is not None else None
    exclude_set = set(exclude_stores)  if exclude_stores  is not None else set()

    # 기존 순서 보존 원하면 sort=False 유지
    for _, g in train_df.groupby(group_col, sort=False):
        store = _extract_store_name(g)

        # 적용 여부 결정
        apply_flag = True
        if apply_set is not None:
            apply_flag = (store in apply_set)
        if store in exclude_set:
            apply_flag = False

        if apply_flag:
            parts.append(remove_leading_zeros_before_sales(g, min_zero_days))
        else:
            parts.append(g)

    if parts:
        return pd.concat(parts, ignore_index=True)
    return train_df.reset_index(drop=True)

# === 추가: Fourier 계절 피처 함수 ===
def add_fourier_seasonal_features(df: pd.DataFrame, date_col: str = '영업일자') -> pd.DataFrame:
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])
    m = df[date_col].dt.month.astype(np.int16)          # 1..12
    doy = df[date_col].dt.dayofyear.astype(np.int16)    # 1..365 (윤년은 무시해도 충분)
    # (A) 월 주기: 12개월 주기, k=1..3 고차 조화항
    for k in (1, 2, 3):
        df[f'month_sin{k}'] = np.sin(2*np.pi*k*m/12).astype('float32')
        df[f'month_cos{k}'] = np.cos(2*np.pi*k*m/12).astype('float32')
    # (B) 연간 주기: 365일 주기, k=1..3 고차 조화항
    for k in (1, 2, 3):
        df[f'doy_sin{k}'] = np.sin(2*np.pi*k*doy/365).astype('float32')
        df[f'doy_cos{k}'] = np.cos(2*np.pi*k*doy/365).astype('float32')
    # (C) 범주형 월 인덱스(임베딩용)
    df['month_idx'] = (m - 1).astype(int)  # 0~11
    return df

#################Loss######################### 

def _slugify(text: str) -> str:
    # 파일명 안전 문자열
    text = re.sub(r'[^\w\-_. ]', '_', text)
    return re.sub(r'\s+', '_', text).strip('_')[:80]

def mse_real_loss(pred_scaled: torch.Tensor, target_scaled: torch.Tensor,
                  y_min: float, y_max: float) -> torch.Tensor:
    """
    pred_scaled, target_scaled: (B, H)  # 0~1 스케일 상
    y_min, y_max: scaler_y.data_min_[0], data_max_[0]
    """
    y_min_t = torch.tensor(float(y_min), device=pred_scaled.device)
    y_max_t = torch.tensor(float(y_max), device=pred_scaled.device)
    pred_real   = pred_scaled  * (y_max_t - y_min_t) + y_min_t
    target_real = target_scaled* (y_max_t - y_min_t) + y_min_t
    return F.mse_loss(pred_real, target_real)

def estimate_delta_from_y(y_real: np.ndarray, clip_min=10.0, clip_max=60.0) -> float:
    # y_real: 실측 스케일(매출수량)
    med = np.median(y_real)
    mad = np.median(np.abs(y_real - med)) + 1e-6   # robust scale
    delta = 1.35 * mad                              # Huber 권장 상수
    return float(np.clip(delta, clip_min, clip_max))

def _to_real(pred_s, tgt_s, y_min, y_max, device):
    y_min_t = torch.tensor(float(y_min), device=device)
    y_max_t = torch.tensor(float(y_max), device=device)
    p = pred_s  * (y_max_t - y_min_t) + y_min_t
    t = tgt_s   * (y_max_t - y_min_t) + y_min_t
    return p, t

def huber_real_weighted(pred_s, tgt_s, y_min, y_max, delta: float, tau: float = 5.0):
    """
    실측 스케일 Huber에 SMAPE-의식적 가중치 적용(타깃 기반, 예측 독립 → 안정).
    w_t = 2 / (|y_t| + tau)
    """
    p, t = _to_real(pred_s, tgt_s, y_min, y_max, device=pred_s.device)
    hub = F.huber_loss(p, t, delta=delta, reduction='none')   # (B,H)
    w = 2.0 / (t.abs() + tau)                                 # (B,H)
    return (hub * w).mean()

def smape_real_loss(pred_s, tgt_s, y_min, y_max, eps=1e-6):
    p, t = _to_real(pred_s, tgt_s, y_min, y_max, device=pred_s.device)
    return ( (p-t).abs() / ((p.abs() + t.abs()).clamp_min(eps) * 0.5) ).mean()

def huber_plus_smape(pred_s, tgt_s, y_min, y_max, delta: float, smape_w: float = 0.15):
    hub = F.huber_loss(*_to_real(pred_s, tgt_s, y_min, y_max, pred_s.device), delta=delta)
    smp = smape_real_loss(pred_s, tgt_s, y_min, y_max)
    return (1.0 - smape_w) * hub + smape_w * smp


def save_mse_curves(history, title="MSE Curve", out_dir="./loss_plots_mse", filename="mse_curve.png"):
    train_mse = list(history.get('train_mse', []))
    val_mse   = list(history.get('val_mse', []))
    L = max(len(train_mse), len(val_mse))
    train_mse += [None]*(L - len(train_mse))
    val_mse   += [None]*(L - len(val_mse))

    plt.figure(figsize=(8,5), dpi=140)
    plt.plot(range(1,L+1), train_mse, marker='o', label='Train MSE')
    plt.plot(range(1,L+1), val_mse,   marker='o', label='Validation MSE')
    plt.title(title); plt.xlabel('Epoch'); plt.ylabel('MSE (real scale)')
    plt.grid(alpha=0.4, linestyle='--'); plt.legend()
    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, filename)
    plt.savefig(path, bbox_inches='tight'); plt.close()
    return path
####################################################

def add_holiday_proximity(
    df: pd.DataFrame,
    date_col: str = '영업일자',
    holiday_col: str = 'is_holiday',
    out_col: str = 'holiday_prox',
    K: int = 7,
    return_what: str = 'prox',  # 'prox' 또는 'dist'
) -> pd.DataFrame:
    """
    캘린더 휴일 기준으로 각 날짜가 휴일에 얼마나 근접했는지 계산합니다.
    - prox: (K - min(dist_prev, dist_next)) / K ∈ [0,1], 당일 휴일=1, K일 이상 떨어지면 0
    - dist: min(dist_prev, dist_next) ∈ [0, K] (K로 클리핑)

    Notes
    -----
    * holiday_col은 미래를 '알 수 있는' 캘린더 정보이므로 누수 위험이 없습니다.
    * df의 원래 행 순서를 유지합니다.
    """
    if date_col not in df.columns:
        raise KeyError(f"'{date_col}' not in df")
    if holiday_col not in df.columns:
        raise KeyError(f"'{holiday_col}' not in df")

    # 원래 인덱스 저장
    orig_index = df.index

    # 날짜 정렬본으로 계산
    tmp = df[[date_col, holiday_col]].copy()
    tmp[date_col] = pd.to_datetime(tmp[date_col])
    tmp = tmp.sort_values(date_col)

    mask = tmp[holiday_col].astype(bool)
    # 휴일이면 그 날짜, 아니면 NaT
    s_h = tmp[date_col].where(mask)

    # 과거/미래 휴일 날짜
    prev_h = s_h.ffill()
    next_h = s_h.bfill()

    # 거리 계산(일수)
    dist_prev = (tmp[date_col] - prev_h).dt.days.astype('float32')
    dist_next = (next_h - tmp[date_col]).dt.days.astype('float32')

    # 휴일이 아예 없을 때 NaN → K+1로 대체
    dist_prev = dist_prev.fillna(K + 1)
    dist_next = dist_next.fillna(K + 1)

    # 최소 거리 후 K로 클리핑
    dist_h = np.minimum(dist_prev, dist_next).clip(0, K).astype('float32')

    if return_what == 'dist':
        out = dist_h
    elif return_what == 'prox':
        # 근접도: 0(멀다) ~ 1(당일 휴일)
        out = ((K - dist_h) / K).astype('float32')
    else:
        raise ValueError("return_what must be 'prox' or 'dist'")

    # 정렬 전 순서로 복원
    out = out.reindex(tmp.index)                # 안전: 이미 tmp와 동일
    out_df = pd.DataFrame({out_col: out}, index=tmp.index)
    out_df = out_df.reindex(orig_index)         # 원래 df 순서로

    # 원본 df에 컬럼으로 추가
    df[out_col] = out_df[out_col].values.astype('float32')
    return df

#Data load
train = pd.read_csv('./train/train.csv')
train = generate_combined_holiday_list(train, solar_md_holidays, lunar_solar_dates)
train = filter_all_menus_by_leading_zeros(
    train,
    min_zero_days=90,
    apply_to_stores=['담하','라그로타','미라시아' ]  # 여기에 대상 업장명만 나열
)
class MRBlock(nn.Module):
    def __init__(self, in_dim, horizon, pool_k: int, mlp_dim=128, mlp_layers=2):
        super().__init__()
        self.pool_k = pool_k                    # e.g., 7, 3, 1
        layers = [nn.Linear(in_dim, mlp_dim), nn.ReLU()]
        for _ in range(mlp_layers-1):
            layers += [nn.Linear(mlp_dim, mlp_dim), nn.ReLU()]
        self.mlp = nn.Sequential(*layers)
        self.proj = nn.Linear(mlp_dim, horizon) # pooled 각 시점→H 로 사영
        self.norm = nn.LayerNorm(mlp_dim)
        self.drop = nn.Dropout(p=0.1)

    def forward(self, x):                       # x: (B,T,F)
        # (B,F,T) → 1D pool → (B,F,T')
        x_ch = x.transpose(1, 2)
        if self.pool_k > 1:
            x_pool = F.avg_pool1d(x_ch, kernel_size=self.pool_k, stride=self.pool_k, ceil_mode=True)
        else:
            x_pool = x_ch
        # (B,T',F)
        x_pool = x_pool.transpose(1, 2)
        z = self.mlp(x_pool)     # (B,T',D)
        z = self.norm(z)
        z = self.drop(z)
        h = self.proj(z)         # (B,T',H)
        h = h.transpose(1, 2).mean(dim=2)  # (B,H)
        return h

class NHiTSWithEmbeddingMR(nn.Module):
    def __init__(self, lookback, input_dim, horizon,
                 pools=(28,7,3,1),                       # <- 28일 스케일 추가
                 weekday_vocab=7, weekday_emb_dim=2,
                 season_vocab=4,  season_emb_dim=2,
                 month_vocab=12,  month_emb_dim=3,       # <- 월 임베딩 추가
                 emb_dropout=0.25, use_sigmoid_output=False):
        super().__init__()
        self.lookback, self.horizon = lookback, horizon
        self.use_sigmoid_output = use_sigmoid_output

        self.weekday_emb = nn.Embedding(weekday_vocab, weekday_emb_dim)
        self.season_emb  = nn.Embedding(season_vocab,  season_emb_dim)
        self.month_emb   = nn.Embedding(month_vocab,   month_emb_dim)   # <- 추가

        self._in_cat = weekday_emb_dim + season_emb_dim + month_emb_dim
        self.post_emb_norm = nn.LayerNorm(input_dim + self._in_cat)
        self.post_emb_drop = nn.Dropout(emb_dropout)

        self.blocks = nn.ModuleList([
            MRBlock(in_dim=input_dim + self._in_cat,
                    horizon=horizon, pool_k=p, mlp_dim=128, mlp_layers=2)
            for p in pools
        ])

    def forward(self, x_num, x_weekday, x_season, x_month):  # <- x_month 추가
        w = self.weekday_emb(x_weekday)   # (B,T,wd_dim)
        s = self.season_emb(x_season)     # (B,T,ss_dim)
        m = self.month_emb(x_month)       # (B,T,mm_dim)
        x = torch.cat([x_num, w, s, m], dim=-1)
        x = self.post_emb_norm(x)
        x = self.post_emb_drop(x)
        y = 0
        for b in self.blocks:
            y = y + b(x)                  # (B,H)
        if self.use_sigmoid_output:
            y = torch.sigmoid(y)
        return y
    
def clip_iqr(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    upper = q3 + 1.5 * iqr
    return np.clip(series, None, upper)

# =========================
# 1) Train: N-HiTS + Embedding 
# =========================
def train_nhits_embed(train_df, use_validation=True,  
                      lr: float = 1e-3, weight_decay: float = 1e-5,
                      max_grad_norm: float = 1.0,
                      plot_dir: str = './loss_plots',
                      use_sigmoid_output: bool = False,   # 스파이크 추종 위해 기본 False 권장
                      ):
    trained_models = {}

    for store_menu, group in tqdm(train_df.groupby(['영업장명_메뉴명']), desc='Training N-HiTS + Emb'):
        # -------- preproc --------
        key = store_menu[0] if isinstance(store_menu, tuple) else store_menu
        store_train = group.sort_values('영업일자').copy()
        store_train['영업일자'] = pd.to_datetime(store_train['영업일자'])
        store_train['weekday']  = store_train['영업일자'].dt.dayofweek.astype(int)    # 0~6
        m = store_train['영업일자'].dt.month.astype(np.int16)
        # 기존 month_sin/month_cos 생성 코드를 제거/주석하고 이 한 줄로 대체
        store_train = add_fourier_seasonal_features(store_train, date_col='영업일자')
        store_train['season']    = m.map({12:0,1:0,2:0, 3:1,4:1,5:1, 6:2,7:2,8:2, 9:3,10:3,11:3}).astype(int)  # 0~3

        t = (store_train['영업일자'] - store_train['영업일자'].min()).dt.days.values
        for k in (1, 2):
            store_train[f'w_sin{k}'] = np.sin(2*np.pi*k*t/7).astype('float32')
            store_train[f'w_cos{k}'] = np.cos(2*np.pi*k*t/7).astype('float32')

        store_train = add_holiday_proximity(store_train, '영업일자', 'is_holiday', 'holiday_prox', K=7, return_what='prox')
        for k in (1,2,3):
            store_train[f'holiday_prox_lag{k}']  = store_train['holiday_prox'].shift(k).fillna(0).astype('float32')
            store_train[f'holiday_prox_lead{k}'] = store_train['holiday_prox'].shift(-k).fillna(0).astype('float32')

        # 타깃 파생
        store_train['clipped_SQ']     = clip_iqr(store_train['매출수량'])
        store_train['clipped_SQ_raw'] = clip_iqr(store_train['매출수량'])
        store_train['delta']          = store_train['clipped_SQ'].diff().fillna(0)
        store_train['rolling_mean_7'] = store_train['clipped_SQ'].rolling(window=7, min_periods=1).mean()

        if len(store_train) < LOOKBACK + PREDICT + MIN_SEQUENCE_COUNT:
            continue

        # -------- split 기준 --------
        N = len(store_train)
        if use_validation:
            cutoff_row = max(LOOKBACK, int(round(N * 0.8)))
            if cutoff_row >= N:
                cutoff_row = N - 1
        else:
            cutoff_row = N

        # -------- 스케일링 --------
        scaler_y     = MinMaxScaler()
        scaler_rm    = MinMaxScaler()
        scaler_delta = MinMaxScaler()
        train_slice  = slice(0, cutoff_row)

        if cutoff_row < 1:
            continue

        store_train['clipped_SQ']     = store_train['clipped_SQ'].astype('float32')
        store_train['rolling_mean_7'] = store_train['rolling_mean_7'].astype('float32')
        store_train['delta']          = store_train['delta'].astype('float32')

        scaler_y.fit(     store_train[['clipped_SQ']].iloc[train_slice] )
        scaler_rm.fit(    store_train[['rolling_mean_7']].iloc[train_slice] )
        scaler_delta.fit( store_train[['delta']].iloc[train_slice] )

        store_train.loc[:, 'clipped_SQ']     = scaler_y.transform(    store_train[['clipped_SQ']].iloc[:N]).ravel().astype('float32')
        store_train.loc[:, 'rolling_mean_7'] = scaler_rm.transform(   store_train[['rolling_mean_7']].iloc[:N]).ravel().astype('float32')
        store_train.loc[:, 'delta_scaled']   = scaler_delta.transform(store_train[['delta']].iloc[:N]).ravel().astype('float32')

        y_train_real = group.sort_values('영업일자')['매출수량'].iloc[:cutoff_row].values
        delta_y = estimate_delta_from_y(y_train_real)   # 예: 10~60 사이로 클리핑됨

        # 통계 피처
        store_train = add_ts_stats(store_train, target_col="clipped_SQ", date_col="영업일자",
                                   lags=(1,7,14,28), roll_windows=(7,14,28), ewm_spans=(7,14))
        nan_cols = [c for c in store_train.columns if c.startswith(('lag_','momentum_'))]
        store_train[nan_cols] = store_train[nan_cols].fillna(0.0).astype('float32')

        # 수치 피처(임베딩 제외)
        num_features = [
            'clipped_SQ','rolling_mean_7','delta_scaled',
            # 월/연 Fourier (k=1..3)
            'month_sin1','month_cos1','month_sin2','month_cos2','month_sin3','month_cos3',
            'doy_sin1','doy_cos1','doy_sin2','doy_cos2','doy_sin3','doy_cos3',
            'holiday_prox','is_holiday',
            'w_sin1','w_sin2','w_cos1','w_cos2',
            'lag_7','lag_14','lag_28',
            'rel_level_7','rel_level_14',
            'vol_7','vol_14',
            'momentum_7','momentum_14',
            'ewm_mean_7','ewm_mean_14',
            'holiday_prox_lag1', 'holiday_prox_lag2', 'holiday_prox_lag3',
            'holiday_prox_lead1', 'holiday_prox_lead2', 'holiday_prox_lead3'
        ]
        store_train[num_features] = store_train[num_features].fillna(0.0).astype('float32')


        # ---------- 시퀀스 ----------
        Xs, Ys, Wd, Ss, Mm, seq_targets = [], [], [], [], [], []
        start_i = MAX_LAG
        end_i   = len(store_train) - LOOKBACK - PREDICT + 1
        for i in range(start_i, end_i):
            X_num = store_train[num_features].values[i:i+LOOKBACK].astype('float32')
            y_seq = store_train['clipped_SQ'].values[i+LOOKBACK:i+LOOKBACK+PREDICT].astype('float32')
            if not np.isfinite(X_num).all() or not np.isfinite(y_seq).all(): continue
            Xs.append(X_num); Ys.append(y_seq)
            Wd.append(store_train['weekday'].values[i:i+LOOKBACK])
            Ss.append(store_train['season'].values[i:i+LOOKBACK])
            Mm.append(store_train['month_idx'].values[i:i+LOOKBACK])   # <- 추가
            seq_targets.append(i + LOOKBACK + PREDICT - 1)
        if not Xs: continue

        X_num = torch.tensor(np.stack(Xs)).float()     # (S,T,F)
        y     = torch.tensor(np.stack(Ys)).float()     # (S,H)
        wd    = torch.tensor(np.stack(Wd)).long()      # (S,T)
        ss    = torch.tensor(np.stack(Ss)).long()      # (S,T)
        mm = torch.tensor(np.stack(Mm)).long()     # <- 추가
        


        # ---------- split tensors ----------
        use_val = use_validation  # 메뉴별 로컬 복사
        if use_val:
            seq_targets = np.asarray(seq_targets)
            train_mask  = seq_targets < cutoff_row
            val_mask    = ~train_mask
            if train_mask.sum() == 0 or val_mask.sum() == 0:
                Xtr, ytr, wdtr, sstr, mmtr = X_num, y, wd, ss, mm
                use_val = False
            else:
                Xtr, Xval = X_num[train_mask], X_num[val_mask]
                ytr, yval = y[train_mask],     y[val_mask]
                wdtr, wdval = wd[train_mask],  wd[val_mask]
                sstr, ssval = ss[train_mask],  ss[val_mask]
                mmtr, mmval = mm[train_mask],  mm[val_mask]
        else:
            Xtr, ytr, wdtr, sstr, mmtr = X_num, y, wd, ss, mm

        # -------- device ----------
        Xtr, ytr = Xtr.to(DEVICE), ytr.to(DEVICE)
        wdtr, mmtr, sstr = wdtr.to(DEVICE), mmtr.to(DEVICE), sstr.to(DEVICE)
        if use_val:
            Xval, yval = Xval.to(DEVICE), yval.to(DEVICE)
            wdval, mmval, ssval = wdval.to(DEVICE), mmval.to(DEVICE), ssval.to(DEVICE)

        model = NHiTSWithEmbeddingMR(
            lookback=LOOKBACK, input_dim=len(num_features), horizon=PREDICT,
            pools=(28,7,3,1),
            weekday_vocab=7, weekday_emb_dim=2,
            season_vocab=4,  season_emb_dim=2,
            month_vocab=12,  month_emb_dim=3,   # <- 추가
            use_sigmoid_output=use_sigmoid_output,
            emb_dropout=0.25
        ).to(DEVICE)

        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

        # ---------- 로깅 ----------
        train_, val_ = [], []
        early = EarlyStopping(patience=PATIENCE, min_delta=MIN_DELTA, mode='min', restore_best_weights=True)
        best_state = None
        best_val = float('inf')

        # 스토어별 y_min/y_max 텐서 (원스케일 SMAPE 계산용)
        # scaler_y.data_min_/data_max_는 shape (1,) 또는 (d,)
        y_min_val = float(scaler_y.data_min_[0]); y_max_val = float(scaler_y.data_max_[0])


        for ep in range(EPOCHS):
            model.train()
            idx = torch.randperm(len(Xtr))
            sum = 0.0; n_obs = 0
            NOISE_BLOCK = {
                # 결정론/이진
                'is_holiday','holiday_prox',
                'holiday_prox_lag1','holiday_prox_lag2','holiday_prox_lag3',
                'holiday_prox_lead1','holiday_prox_lead2','holiday_prox_lead3',
                # 주기(결정론)
                'w_sin1','w_sin2','w_cos1','w_cos2',
                'month_sin1','month_cos1','month_sin2','month_cos2','month_sin3','month_cos3',
                'doy_sin1','doy_cos1','doy_sin2','doy_cos2','doy_sin3','doy_cos3',
            }
            noise_idx = [j for j,c in enumerate(num_features) if c not in NOISE_BLOCK]

            # 에폭별 노이즈 스케줄(처음엔 0.02, 마지막엔 0)
            base_noise = 0.02
            noise_std = base_noise * (1.0 - ep / max(EPOCHS, 1))  # 선형 감쇠

            for i in range(0, len(Xtr), BATCH_SIZE):
                bidx = idx[i:i+BATCH_SIZE]
                Xb, yb = Xtr[bidx], ytr[bidx]
                wdb, mmb, ssb = wdtr[bidx], mmtr[bidx], sstr[bidx]

                # ✅ 연속형 피처에만 노이즈 주입
                if noise_idx and noise_std > 0:
                    Xb[..., noise_idx] = Xb[..., noise_idx] + noise_std * torch.randn_like(Xb[..., noise_idx])


                pred = model(Xb, wdb, ssb, mmb)
                loss = huber_real_weighted(pred, yb, y_min_val, y_max_val, delta=delta_y, tau=5.0)


                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                if max_grad_norm is not None:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
                optimizer.step()

                bs = yb.size(0)
                sum += loss.item() * bs
                n_obs += bs

            train_.append(sum / max(n_obs,1))

            if use_val:
                model.eval()
                with torch.no_grad():
                    sum= 0.0; n_obs = 0
                    for i in range(0, len(Xval), BATCH_SIZE):
                        Xb, yb = Xval[i:i+BATCH_SIZE], yval[i:i+BATCH_SIZE]
                        wdb, ssb = wdval[i:i+BATCH_SIZE], ssval[i:i+BATCH_SIZE]
                        mmb      = mmval[i:i+BATCH_SIZE]
                        pred = model(Xb, wdb, ssb, mmb)
                        loss = huber_real_weighted(pred, yb, y_min_val, y_max_val, delta=delta_y, tau=5.0)
                        bs = yb.size(0)
                        sum += loss.item() * bs
                        n_obs += bs
                val_.append(sum / max(n_obs,1))

                # ---- EarlyStop / LR 스케줄러: 모두 Val MSE 기준 ----
                stop_loss = val_[-1]
                scheduler.step(stop_loss)
                if early.step(stop_loss, model):
                    print(f"[{key}] Early stop @ {ep+1} | best val_mse={min(val_):.6f}")
                    break

                if val_[-1] < best_val - 1e-12:
                    best_val = val_[-1]
                    best_state = {k: v.detach().cpu() for k,v in model.state_dict().items()}
            else:
                val_.append(train_[-1])

        if best_state is not None:
            model.load_state_dict(best_state)

        # ---- 플롯 저장 ----
        history = {'train_mse': train_, 'val_mse': val_}
        try:
            os.makedirs(plot_dir, exist_ok=True)
            fname = f"{_slugify(str(key))}_mse_curve.png"
            save_mse_curves(history, title=f"{key} MSE Curve", out_dir=plot_dir, filename=fname)
        except Exception as e:
            print(f"[{key}] plot save skipped: {e}")

        # ---- 상한선 저장 & 시퀀스 캐시 ----
        lower_bound = float(max(np.quantile(group['매출수량'].values, 0.05), 1.0))
        upper_bound = float(np.quantile(group['매출수량'].values, 0.995))

        trained_models[store_menu] = {
            'model': model.eval(),
            'scaler_y': scaler_y, 'scaler_rm': scaler_rm, 'scaler_delta': scaler_delta,
            'upper_bound': upper_bound,
            'lower_bound' : lower_bound,
            'feature_order': num_features,
            'history': history,
            'last_sequence': {
                'X_num': store_train[num_features].values[-LOOKBACK:],
                'weekday': store_train['weekday'].values[-LOOKBACK:],
                'season':  store_train['season'].values[-LOOKBACK:],
                'month_idx': store_train['month_idx'].values[-LOOKBACK:]   # <- 추가
            }
        }

    return trained_models


def visualize_loss(
    train_losses,
    val_losses=None,
    store_menu="",
    save=False,
    out_dir="./loss_plots",
    use_smape=True,           # True면 % 단위로 표시 (SMAPE)
    early_stop_epoch=None,    # EarlyStopping으로 멈춘 에폭(0-index). 없으면 자동 탐지
    title_prefix="[Loss]"
):
    """
    train_losses, val_losses: 에폭별 loss 리스트
    use_smape=True  -> y축을 %로 표기하고 값*100으로 시각화
    early_stop_epoch:
        - None이면 val 최소값 위치(최고 성능)를 auto로 마킹
        - 정수를 주면 해당 에폭에 수직선 표시
    """

    # 안전 가드
    train_arr = np.asarray(train_losses, dtype=float)
    val_arr   = None if val_losses is None else np.asarray(val_losses, dtype=float)

    # SMAPE면 퍼센트 스케일로 변환
    if use_smape:
        plot_train = train_arr * 100.0
        plot_val   = None if val_arr is None else val_arr * 100.0
        y_label = "SMAPE (%)"
    else:
        plot_train = train_arr
        plot_val   = None if val_arr is None else val_arr
        y_label = "Loss"

    plt.figure()
    plt.plot(plot_train, label='Train', linewidth=2)

    if plot_val is not None:
        plt.plot(plot_val, label='Validation', linewidth=2)

        # 최고 성능(= 최소 val) 에폭 표시 또는 전달된 early_stop_epoch 사용
        if early_stop_epoch is None:
            best_epoch = int(np.argmin(val_arr))  # 원본 스케일 기준
        else:
            best_epoch = int(early_stop_epoch)

        plt.axvline(best_epoch, linestyle='--', alpha=0.5, label=f'Best @ {best_epoch}')
        # 그 지점 값에 마커
        plt.scatter([best_epoch], [plot_val[best_epoch]], zorder=3)

    title_core = "SMAPE" if use_smape else "Loss"
    plt.title(f"{title_prefix} [{store_menu}] Train vs Validation ({title_core})")
    plt.xlabel("Epoch")
    plt.ylabel(y_label)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    if save:
        os.makedirs(out_dir, exist_ok=True)
        safe_name = re.sub(r'[^\w\-_.]', '_', str(store_menu))
        path = os.path.join(out_dir, f"{safe_name}.png")
        plt.savefig(path, dpi=150)
    else:
        plt.show()

    plt.close()


def predict_nhits_embed(
    test_df,
    trained_models,
    test_prefix: str,
    *,
    discontinued: dict[str, str | pd.Timestamp] | None = None,  # <- 추가
    rule: str = 'after',     # 'after' : 단종일자 이후(>) 0, 'on_or_after' : 단종일자 당일 포함(>=) 0
    grace_days: int = 0      # 유예일 (단종일자 + grace_days 이후부터 0)
):
    """
    N-HiTS(+weekday/season 임베딩) 예측 파이프라인 (+ 단종 처리)
    - 입력: test_df (영업일자, 영업장명_메뉴명, 매출수량 등 포함)
    - 출력: pd.DataFrame([영업일자, 영업장명_메뉴명, 매출수량])
    - 주의: feature 목록/순서는 train_nhits_embed와 동일해야 함.
    - discontinued: {'영업장명_메뉴명': 'YYYY-MM-DD' 또는 Timestamp} 형태의 단종 딕셔너리
    """
    results = []

    num_features = [
            'clipped_SQ','rolling_mean_7','delta_scaled',
            # 월/연 Fourier (k=1..3)
            'month_sin1','month_cos1','month_sin2','month_cos2','month_sin3','month_cos3',
            'doy_sin1','doy_cos1','doy_sin2','doy_cos2','doy_sin3','doy_cos3',
            'holiday_prox','is_holiday',
            'w_sin1','w_sin2','w_cos1','w_cos2',
            'lag_7','lag_14','lag_28',
            'rel_level_7','rel_level_14',
            'vol_7','vol_14',
            'momentum_7','momentum_14',
            'ewm_mean_7','ewm_mean_14',
            'holiday_prox_lag1', 'holiday_prox_lag2', 'holiday_prox_lag3',
            'holiday_prox_lead1', 'holiday_prox_lead2', 'holiday_prox_lead3'
        ]

    # ----- (A) 단종 딕셔너리 Timestamp 정규화 -----
    cutoff_map = None
    if discontinued is not None:
        def _to_ts(v):
            return v if isinstance(v, pd.Timestamp) else pd.to_datetime(v)
        cutoff_map = {k: _to_ts(v) for k, v in discontinued.items()}
        if grace_days != 0:
            for k in cutoff_map:
                cutoff_map[k] = cutoff_map[k] + pd.Timedelta(days=grace_days)

    for store_menu, store_test in test_df.groupby(['영업장명_메뉴명']):
        key = store_menu
        if key not in trained_models:
            continue

        pack         = trained_models[key]
        model        = pack['model']
        scaler_y     = pack['scaler_y']
        scaler_rm    = pack['scaler_rm']
        scaler_delta = pack['scaler_delta']
        upper_bound  = pack.get('upper_bound', None)
        lower_bound  = pack.get('lower_bound', 1.0)

        # 1) 정렬/파생 & 캘린더 파트
        store_test_sorted = store_test.sort_values('영업일자').copy()
        store_test_sorted['영업일자'] = pd.to_datetime(store_test_sorted['영업일자'])
        store_test_sorted['weekday']  = store_test_sorted['영업일자'].dt.dayofweek.astype(int)  # 0~6
        m = store_test_sorted['영업일자'].dt.month.astype(np.int16)
        # Fourier + month_idx 생성
        store_test_sorted = add_fourier_seasonal_features(store_test_sorted, date_col='영업일자')

        store_test_sorted['season']    = m.map({12:0,1:0,2:0, 3:1,4:1,5:1, 6:2,7:2,8:2, 9:3,10:3,11:3}).astype(int)  # 0~3
        t = (store_test_sorted['영업일자'] - store_test_sorted['영업일자'].min()).dt.days.values
        for k in (1, 2):
            store_test_sorted[f'w_sin{k}'] = np.sin(2*np.pi*k*t/7).astype('float32')
            store_test_sorted[f'w_cos{k}'] = np.cos(2*np.pi*k*t/7).astype('float32')

        # 공휴일/근접도 (학습과 동일 함수 사용)
        store_test_sorted = generate_combined_holiday_list(store_test_sorted, solar_md_holidays, lunar_solar_dates)
        store_test_sorted = add_holiday_proximity(
            store_test_sorted, date_col='영업일자', holiday_col='is_holiday',
            out_col='holiday_prox', K=7, return_what='prox'
        )
        for k in (1,2,3):
            store_test_sorted[f'holiday_prox_lag{k}']  = store_test_sorted['holiday_prox'].shift(k).fillna(0).astype('float32')
            store_test_sorted[f'holiday_prox_lead{k}'] = store_test_sorted['holiday_prox'].shift(-k).fillna(0).astype('float32')

        # 2) 타깃 파생(관측기반)
        store_test_sorted['clipped_SQ']     = clip_iqr(store_test_sorted['매출수량'])
        store_test_sorted['rolling_mean_7'] = store_test_sorted['clipped_SQ'].rolling(window=7, min_periods=1).mean()
        store_test_sorted['delta']          = store_test_sorted['clipped_SQ'].diff().fillna(0)

        # 4) 스케일링 (학습 스케일러로 동일 순서 적용)
        store_test_sorted[['delta_scaled']]   = scaler_delta.transform(store_test_sorted[['delta']])
        store_test_sorted[['clipped_SQ']]     = scaler_y.transform(store_test_sorted[['clipped_SQ']])
        store_test_sorted[['rolling_mean_7']] = scaler_rm.transform(store_test_sorted[['rolling_mean_7']])

        # 3) 통계 피처 (lag/roll/ewm 등)
        store_test_sorted = add_ts_stats(
            store_test_sorted, target_col='clipped_SQ', date_col='영업일자',
            lags=(1,7,14,28), roll_windows=(7,14,28), ewm_spans=(7,14)
        )
        nan_cols = [c for c in store_test_sorted.columns if c.startswith(('lag_','momentum_'))]
        store_test_sorted[nan_cols] = store_test_sorted[nan_cols].fillna(0.0).astype('float32')
        safe_cols = list(set(num_features) & set(store_test_sorted.columns))
        store_test_sorted[safe_cols] = store_test_sorted[safe_cols].fillna(0.0).astype('float32')

        # 5) 입력 윈도우 구성
        if len(store_test_sorted) < LOOKBACK:
            last_seq  = pack['last_sequence']
            x_num_np  = np.asarray(last_seq['X_num'], dtype='float32')
            weekday_np= np.asarray(last_seq['weekday'], dtype='int64')
            season_np = np.asarray(last_seq['season'],  dtype='int64')
            month_np  = np.asarray(last_seq['month_idx'], dtype='int64')  # <- 추가
            last_obs_date = pd.to_datetime(store_test_sorted['영업일자'].max())
        else:
            recent = store_test_sorted.iloc[-LOOKBACK:].copy()
            for col in num_features:
                if col not in recent.columns:
                    recent[col] = 0.0
            recent = recent[num_features].astype('float32')

            x_num_np   = recent.values
            weekday_np = store_test_sorted['weekday'].values[-LOOKBACK:].astype('int64')
            season_np  = store_test_sorted['season'].values[-LOOKBACK:].astype('int64')
            month_np   = store_test_sorted['month_idx'].values[-LOOKBACK:].astype('int64')  # <- 추가
            last_obs_date = pd.to_datetime(store_test_sorted['영업일자'].max())

        # 텐서 변환 시 x_month 포함
        x_num_input = torch.tensor(x_num_np, dtype=torch.float32, device=DEVICE).unsqueeze(0)
        weekday_seq = torch.tensor(weekday_np, dtype=torch.long,   device=DEVICE).unsqueeze(0)
        season_seq  = torch.tensor(season_np,  dtype=torch.long,   device=DEVICE).unsqueeze(0)
        month_seq   = torch.tensor(month_np,   dtype=torch.long,   device=DEVICE).unsqueeze(0)   # <- 추가

          # 6) 예측
        model.eval()
        with torch.no_grad():
            pred_scaled = model(x_num_input, weekday_seq, season_seq, month_seq).squeeze(0).cpu().numpy()

        # 7) 역정규화 전 clip 여부
        use_sigmoid = getattr(model, 'use_sigmoid_output', False)
        if use_sigmoid:
            pred_scaled = np.clip(pred_scaled, 0.0, 1.0)

    
        vals_real   = scaler_y.inverse_transform(pred_scaled.reshape(-1,1)).ravel()

        # 8) 상/하한 clip
        if upper_bound is not None:
            vals_real = np.clip(vals_real, lower_bound, upper_bound)
        else:
            vals_real = np.clip(vals_real, lower_bound, None)

        # ----- (B) 단종 처리: 예측 구간 실제 날짜와 비교 -----
        # 예측 구간의 "실제 달력 날짜" 생성 (last_obs_date 다음날부터 PREDICT일)
        horizon_dates = pd.date_range(start=last_obs_date + pd.Timedelta(days=1),
                                      periods=PREDICT, freq='D')

        if cutoff_map is not None:
            cutoff = cutoff_map.get(key, None)
            if cutoff is not None:
                if rule == 'on_or_after':
                    zero_mask = horizon_dates >= cutoff
                else:  # 'after'
                    zero_mask = horizon_dates > cutoff
                vals_real = np.where(zero_mask, 0.0, vals_real)

        # 9) 제출 포맷 적재 (네 포맷 그대로)
        pred_dates = [f"{test_prefix}+{i+1}일" for i in range(PREDICT)]
        menu_name = store_menu[0] if isinstance(store_menu, tuple) else store_menu
        for d, v in zip(pred_dates, vals_real):
            results.append({'영업일자': d, '영업장명_메뉴명': menu_name, '매출수량': float(v)})

    return pd.DataFrame(results)




def convert_to_submission_format(pred_df: pd.DataFrame, sample_submission: pd.DataFrame):
    # (영업일자, 메뉴) → 매출수량 딕셔너리로 변환
    pred_dict = dict(zip(
        zip(pred_df['영업일자'], pred_df['영업장명_메뉴명']),
        pred_df['매출수량']
    ))

    final_df = sample_submission.copy()

    for row_idx in final_df.index:
        date = final_df.loc[row_idx, '영업일자']
        for col in final_df.columns[1:]:  # 메뉴명들
            final_df.loc[row_idx, col] = pred_dict.get((date, col), 0)

    return final_df


In [2]:
trained_models = train_nhits_embed(train, use_validation=False, plot_dir='./loss_plots_weighted_huber')

Training N-HiTS + Emb: 100%|██████████| 193/193 [26:56<00:00,  8.38s/it]


In [3]:
all_preds = []

# 모든 test_*.csv 순회
test_files = sorted(glob.glob('./test/TEST_*.csv'))
df = pd.read_csv('./train/train.csv')
for path in test_files:
    test_df = pd.read_csv(path)
    # 파일명에서 접두어 추출 (예: TEST_00)
    filename = os.path.basename(path)
    test_prefix = re.search(r'(TEST_\d+)', filename).group(1)

    pred_df = predict_nhits_embed(
        test_df, trained_models, test_prefix,
        discontinued=DISCONTINUED,
        rule='after',      # 단종일 "이후" 0
        grace_days=0       # 유예일 없으면 0
    )
    all_preds.append(pred_df)
    
full_pred_df = pd.concat(all_preds, ignore_index=True)

In [4]:
sample_submission = pd.read_csv('./sample_submission.csv')
submission = convert_to_submission_format(full_pred_df, sample_submission)
submission.to_csv('./Prediction/model_v6_3.csv', index=False, encoding='utf-8-sig')
result = pd.read_csv('./Prediction/model_v6_3.csv')
display(result.head())

/var/folders/r4/sdnz117n6pl22zr9jhhv5vbm0000gn/T/ipykernel_5054/2470673699.py:1037: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '7.659077167510986' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  final_df.loc[row_idx, col] = pred_dict.get((date, col), 0)
/var/folders/r4/sdnz117n6pl22zr9jhhv5vbm0000gn/T/ipykernel_5054/2470673699.py:1037: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '7.2359700202941895' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  final_df.loc[row_idx, col] = pred_dict.get((date, col), 0)
/var/folders/r4/sdnz117n6pl22zr9jhhv5vbm0000gn/T/ipykernel_5054/2470673699.py:1037: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '3.8073081970214844' has dt

,영업일자,느티나무 셀프BBQ_1인 수저세트,느티나무 셀프BBQ_BBQ55(단체),"느티나무 셀프BBQ_대여료 30,000원","느티나무 셀프BBQ_대여료 60,000원","느티나무 셀프BBQ_대여료 90,000원","느티나무 셀프BBQ_본삼겹 (단품,실내)",느티나무 셀프BBQ_스프라이트 (단체),느티나무 셀프BBQ_신라면,느티나무 셀프BBQ_쌈야채세트,...,화담숲주막_스프라이트,화담숲주막_참살이 막걸리,화담숲주막_찹쌀식혜,화담숲주막_콜라,화담숲주막_해물파전,화담숲카페_메밀미숫가루,화담숲카페_아메리카노 HOT,화담숲카페_아메리카노 ICE,화담숲카페_카페라떼 ICE,화담숲카페_현미뻥스크림
0,TEST_00+1일,7.659077,7.235970,3.807308,1.229030,1.0,1.0,1.000000,1.000000,1.0,...,1.513204,1.000000,1.000000,5.586097,1.000000,1.000000,1.000000,13.544243,2.348771,4.924942
1,TEST_00+2일,4.426912,1.000000,1.000000,1.000000,1.0,1.0,1.000000,1.000000,1.0,...,1.000000,1.915633,3.552394,2.229620,26.393847,2.681314,1.102490,2.562031,4.008732,7.991850
2,TEST_00+3일,7.397885,2.599880,1.000000,1.653031,1.0,1.0,1.904164,1.000000,1.0,...,1.208084,2.974417,1.000000,1.000000,25.917915,6.779230,6.072268,3.263063,4.427794,1.000000
3,TEST_00+4일,6.877163,1.840744,4.165692,1.981568,1.0,1.0,1.499252,1.000000,1.0,...,1.336006,3.335388,5.750009,2.139581,1.000000,5.772413,3.547105,1.000000,3.270935,2.800947
4,TEST_00+5일,4.788754,1.000000,6.324567,1.399796,1.0,1.0,1.000000,1.063058,1.0,...,1.573683,1.000000,1.000000,5.504909,30.245880,11.456173,1.000000,13.545057,2.007706,1.000000
